# 20 - vLLM baseline on the T4 (its own notebook: vLLM installs its own torch)
Run on a **GPU T4 x2** session with Internet on and Persistence = Variables and Files. Press **Run All**; every stage skips work that is already done, so after a crash you can simply press Run All again.

Order: setup, install vLLM, download model/dataset/repo, start the server, ShareGPT sweep (c=1 / c=4 / c=16,32,64, two repetitions each), prefill sweep (2k/4k/8k tokens), bundle the results into one zip. Expect about 45-70 minutes. Progress lines appear after every repetition.

Do not use *Factory reset* (it wipes `/kaggle/working`). Close the laptop lid only after the zip is downloaded.

In [ ]:
import os, sys, json, time, signal, shutil, subprocess, threading, glob

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working"
MODEL = f"{WORK}/models/Qwen2.5-1.5B-Instruct"
REPO = f"{WORK}/turingserve"
SHAREGPT = f"{WORK}/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
RESULTS = f"{WORK}/results"
PORT = 8001
VLLM_EXTRA = []          # e.g. ["--enforce-eager"] if the default settings do not start on the T4
RUN_PREFILL = True       # the 2k/4k/8k prefill sweep needs a server restart with --max-model-len 8192
os.makedirs(RESULTS, exist_ok=True)
os.environ["TS_SHAREGPT"] = SHAREGPT
os.environ["TS_KAGGLE"] = "1"
srv = None


def gpu_used_mib():
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                       capture_output=True, text=True)
    return int(r.stdout.strip().splitlines()[0]) if r.returncode == 0 and r.stdout.strip() else -1


def stop_server():
    global srv
    if srv is not None and srv.poll() is None:
        try:
            os.killpg(os.getpgid(srv.pid), signal.SIGTERM)
            srv.wait(timeout=30)
        except Exception:
            try:
                os.killpg(os.getpgid(srv.pid), signal.SIGKILL)
            except Exception:
                pass
    srv = None
    subprocess.run(["pkill", "-f", "vllm serve"], capture_output=True)   # also clears a server left over from an earlier kernel
    time.sleep(3)
    for _ in range(30):
        if 0 <= gpu_used_mib() < 500:
            break
        time.sleep(2)
    print("server stopped, GPU 0 memory used:", gpu_used_mib(), "MiB")


def start_server(max_len, log_name, extra=()):
    """Launch vLLM, wait for /health, print the first completion."""
    global srv
    import requests
    stop_server()
    log_path = f"{WORK}/{log_name}"
    log = open(log_path, "w")
    srv = subprocess.Popen(["vllm", "serve", MODEL, "--served-model-name", "qwen", "--dtype", "half",
                            "--max-model-len", str(max_len), "--gpu-memory-utilization", "0.85",
                            "--port", str(PORT), *VLLM_EXTRA, *extra],
                           stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    for i in range(240):
        try:
            if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
                print(f"vLLM up after {time.time() - t0:.0f}s (max-model-len {max_len})")
                break
        except Exception:
            pass
        if srv.poll() is not None:
            break
        if i % 12 == 11:
            print(f"  still starting... {time.time() - t0:.0f}s")
        time.sleep(5)
    else:
        print(open(log_path).read()[-3000:])
        raise RuntimeError("vLLM did not become healthy in 20 minutes")
    if srv.poll() is not None:
        print(open(log_path).read()[-3000:])
        raise RuntimeError("vLLM exited during startup, log tail above")
    r = requests.post(f"http://localhost:{PORT}/v1/completions",
                      json={"model": "qwen", "prompt": "The capital of France is", "max_tokens": 16, "temperature": 0})
    print("test completion:", r.json()["choices"][0]["text"])


def run_bench(label, extra):
    """Run serve_bench, streaming its output so progress is visible; a heartbeat prints every 2 minutes."""
    cmd = [sys.executable, "-u", "-m", "benchmarks.serve_bench", "--base-url", f"http://localhost:{PORT}",
           "--engine", "vllm", "--model", "qwen", "--tokenizer", MODEL, "--out", RESULTS, "--label", label, *extra]
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    t0 = time.time()
    done = threading.Event()

    def beat():
        while not done.wait(120):
            print(f"  ... {label} still running ({(time.time() - t0) / 60:.0f} min)", flush=True)

    threading.Thread(target=beat, daemon=True).start()
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Token indices sequence length" in line or "Running this sequence" in line:
            continue
        print(line, end="", flush=True)
    p.wait()
    done.set()
    print(f"[{label}] finished in {(time.time() - t0) / 60:.1f} min, exit code {p.returncode}\n")
    return p.returncode


def result_exists(label):
    return bool(glob.glob(f"{RESULTS}/*_vllm_*_{label}.json"))


print("setup ok; GPU 0 memory used:", gpu_used_mib(), "MiB")

## 1. Install vLLM and record versions
Paste the printed line into `docs/baselines.md`.

In [ ]:
def can_import(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not can_import("vllm"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError("pip install vllm failed")
# the pip warnings about numba, gradio, pillow and opentelemetry concern other preinstalled packages and are harmless
r = subprocess.run([sys.executable, "-c", "import vllm, torch; print(vllm.__version__, torch.__version__)"],
                   capture_output=True, text=True)
print("vllm / torch:", r.stdout.strip() or r.stderr[-1500:])
open(f"{WORK}/vllm_versions.txt", "w").write(r.stdout)

## 2. Model, dataset and benchmark client
The benchmark client is the same `serve_bench` at tag `m3-rc` that produced the TuringServe numbers.

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

if not os.path.exists(f"{MODEL}/model.safetensors"):
    snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=MODEL)
print("model ok:", os.path.exists(f"{MODEL}/model.safetensors"))

if not os.path.exists(SHAREGPT):
    hf_hub_download("anon8231489123/ShareGPT_Vicuna_unfiltered", "ShareGPT_V3_unfiltered_cleaned_split.json",
                    repo_type="dataset", local_dir=f"{WORK}/datasets/sharegpt")
print("dataset ok:", os.path.exists(SHAREGPT))

if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "https://github.com/Muskanpandey0304/turingserve.git", REPO], check=True)
subprocess.run(["git", "fetch", "--tags", "--force"], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "-f", "m3-rc"], cwd=REPO, check=True, capture_output=True)
print("repo at:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO, capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "datasets"], capture_output=True)

## 3. Start vLLM (port 8001)
If it fails to start, the log tail is printed above the error. Paste it and stop; a failure or slow attention path on the T4 is itself a result.

In [ ]:
start_server(4096, "vllm.log")
print("--- attention / backend lines from the vLLM log (record these in docs/baselines.md) ---")
for line in open(f"{WORK}/vllm.log"):
    if any(k in line.lower() for k in ("backend", "attention", "flash", "xformers", "triton", "cuda graph", "compil")):
        print(line.rstrip()[:220])

## 4. ShareGPT sweep

In [ ]:
# c=1 is the slow level (requests run one at a time), so it uses fewer prompts; the high levels keep large samples.
STAGES = [
    ("c1",  ["--num-prompts", "60",  "--concurrency", "1"]),
    ("c4",  ["--num-prompts", "150", "--concurrency", "4"]),
    ("hi",  ["--num-prompts", "400", "--concurrency", "16,32,64"]),
]
COMMON = ["--dataset", "sharegpt", "--mode", "concurrency", "--repeat", "2", "--ignore-eos", "--gpu-log"]
failed = []
for name, extra in STAGES:
    label = f"t4-1.5b-{name}"
    if result_exists(label):
        print(f"[{label}] result already exists, skipping")
        continue
    if run_bench(label, [*COMMON, *extra]) != 0:
        failed.append(label)
print("stages that failed:", failed or "none")

## 5. Prefill sweep (restarts the server with --max-model-len 8192)

In [ ]:
label = "t4-1.5b-prefill-cold"
if not RUN_PREFILL:
    print("prefill skipped (RUN_PREFILL = False)")
elif result_exists(label):
    print(f"[{label}] result already exists, skipping")
else:
    start_server(8192, "vllm_prefill_cold.log", extra=["--no-enable-prefix-caching"])
    run_bench(label, ["--dataset", "prefill", "--num-prompts", "500", "--mode", "concurrency",
                      "--concurrency", "1", "--repeat", "3", "--gpu-log"])

## 6. Summary and download
Download **`/kaggle/working/vllm_bundle.zip`** (results, logs and versions in one file), then stop the session.

In [ ]:
stop_server()

print(f"{'file':58}{'c':>4}{'ok/n':>9}{'err':>5}{'tok/s':>9}{'ttft p50 ms':>13}{'tpot p50 ms':>13}")
for f in sorted(glob.glob(f"{RESULTS}/*_vllm_*.json")):
    d = json.load(open(f))
    for r in d["runs"]:
        m = r["median"]
        p50 = lambda t: (t.get("p50") or 0) * 1000
        print(f"{os.path.basename(f)[:57]:58}{r['level']:>4}{m['num_ok']:>5}/{m['num_requests']:<3}{m['errors']:>5}"
              f"{m['output_throughput_tok_s']:>9.1f}{p50(m['ttft']):>13.1f}{p50(m['tpot']):>13.1f}")

bundle = f"{WORK}/vllm_bundle"
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for f in glob.glob(f"{RESULTS}/*_vllm_*.json") + glob.glob(f"{WORK}/vllm*.log") + [f"{WORK}/vllm_versions.txt"]:
    if os.path.exists(f):
        shutil.copy(f, bundle)
shutil.make_archive(f"{WORK}/vllm_bundle", "zip", bundle)
print("\nDownload this one file:", f"{WORK}/vllm_bundle.zip", f"({os.path.getsize(WORK + '/vllm_bundle.zip') / 1e6:.1f} MB)")